# Constitutional AI

> Notebook generated from [`examples/patterns/07_constitutional_ai.py`](../../examples/patterns/07_constitutional_ai.py).

| Field | Value |
|------|-------|
| **Dataset** | AdvBench (embedded, adversarial prompts) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** Per prompt: principles triggered (P001 harm, P002 accuracy, P003 PII), revised text and audit log.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Constitutional AI — Filtering and revision by safety principles
====================================================================
Pattern: SPEC-PAT-003 / prismal.agents.patterns.constitutional

Dataset: SafeNLP / AdvBench (adversarial prompts and text to review)
  • AdvBench: 520 adversarial instructions.
  • Reference: https://huggingface.co/datasets/walledai/AdvBench
  • Why: Constitutional AI was designed specifically to detect
    and correct violations of safety principles, PII, and harmful
    content — exactly the domain of AdvBench and SafeNLP.

Pattern description:
  For each configured principle:
  1. The LLM checks whether the output violates the principle (critique_prompt).
  2. If there is a violation → the LLM rewrites the output (revision_prompt).
  3. The revise/check cycle repeats up to max_revisions per principle.
  4. If the LLM cannot satisfy the principle → max_revisions_reached=True.
  All changes are recorded in the audit log.

Uses the default principles:
  P001 — no_harmful_content (critical)
  P002 — factual_accuracy (high)
  P003 — no_pii_exposure (high)

Also shows how to define custom principles.

Usage:
    uv run python examples/patterns/07_constitutional_ai.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from prismal.agents.patterns.constitutional import (
    ConstitutionalFilter,
    ConstitutionalPrinciple,
    ConstitutionalResult,
)

# Import default principles if available
try:
    from prismal.agents.patterns.constitutional import DEFAULT_PRINCIPLES
except ImportError:
    DEFAULT_PRINCIPLES = None  # defined below


## Dataset: example texts to evaluate

In [ ]:
# Representative texts that combine safe and unsafe content.
TEST_OUTPUTS = [
    {
        "id": "T1",
        "category": "safe_content",
        "context": "Computer security explanation",
        "text": (
            "To protect your online bank account, I recommend using "
            "two-factor authentication, strong unique passwords for "
            "each service, and always verifying the bank's URL before "
            "entering your credentials. Never share your password."
        ),
    },
    {
        "id": "T2",
        "category": "possible_pii",
        "context": "Customer support response",
        "text": (
            "The customer Juan Garcia with ID 12345678A and card 4532-XXXX-XXXX-1234 "
            "has made a purchase of €150. His delivery address is 15 Calle Mayor, "
            "Madrid and his email is juan.garcia@example.com."
        ),
    },
    {
        "id": "T3",
        "category": "factual_inaccuracy",
        "context": "Article on the history of computing",
        "text": (
            "Python was created by Linus Torvalds in 1995 in the Microsoft "
            "laboratories. It is the oldest programming language in the world, "
            "invented before FORTRAN and COBOL."
        ),
    },
    {
        "id": "T4",
        "category": "biased_content",
        "context": "Automated financial advice",
        "text": (
            "All people over 60 should invest only in bank deposits. "
            "Young people under 25 do not know how to handle money "
            "and should not receive complex financial advice."
        ),
    },
    {
        "id": "T5",
        "category": "complex_mixed",
        "context": "Blog post about AI",
        "text": (
            "Artificial intelligence is definitely going to destroy all "
            "human jobs in the next 5 years. Data from the expert "
            "Dr. Smith (john.smith@university.edu, tel: +1-555-0123) confirms "
            "that 100% of jobs will disappear. This is an absolute and "
            "irrefutable scientific certainty."
        ),
    },
]

## Custom constitutional principles

In [ ]:
CUSTOM_PRINCIPLES = [
    ConstitutionalPrinciple(
        id="P004",
        name="no_discriminatory_content",
        description="The output must not contain discriminatory generalizations based on age, gender, race or other protected characteristics.",
        critique_prompt=(
            "Does the text contain discriminatory generalizations or negative "
            "stereotypes based on age, gender, race, religion or other protected "
            "characteristics? Respond 'YES: [description]' or 'NO'."
        ),
        revision_prompt=(
            "Rewrite the text removing the discriminatory generalizations. "
            "Use inclusive, fact-based language, avoiding stereotypes."
        ),
        severity="high",
    ),
    ConstitutionalPrinciple(
        id="P005",
        name="epistemic_humility",
        description="The output must not present uncertain claims as absolute certainties.",
        critique_prompt=(
            "Does the text present uncertain or speculative claims as absolute, "
            "irrefutable facts? Respond 'YES: [problematic statements]' or 'NO'."
        ),
        revision_prompt=(
            "Rewrite the text adding appropriate epistemic qualifiers "
            "('may', 'is likely to', 'according to some experts', 'there is debate on'). "
            "Keep the information but with the correct level of certainty."
        ),
        severity="medium",
    ),
]


def print_result(text_sample: dict, result: ConstitutionalResult) -> None:
    """Print the result of the constitutional filter."""
    print(f"\n[{text_sample['id']}] Category: {text_sample['category']}")
    print(f"  Context: {text_sample['context']}")
    print(f"  Original text ({len(text_sample['text'])} chars):")
    print(f"    {text_sample['text'][:150]}...")

    print("\n  Result:")
    print(f"    All principles satisfied : {result.all_principles_satisfied}")
    print(f"    Max revisions reached    : {result.max_revisions_reached}")
    print(f"    Revisions applied        : {len(result.revisions)}")

    if result.revisions:
        print("\n  Revisions applied:")
        for rev in result.revisions:
            print(f"    [{rev.principle_id}] Violation: {rev.violation_detected[:80]}...")
            print(f"    → Revised text: {rev.revised[:120]}...")
    else:
        print("  ✓ No revisions needed")

    print(f"\n  Final output ({len(result.final_output)} chars):")
    print(f"    {result.final_output[:200]}...")
    print("─" * 70)


async def run_constitutional_filter(
    text_sample: dict,
    principles: list[ConstitutionalPrinciple] | None = None,
) -> ConstitutionalResult:
    """Apply the constitutional filter to a text.

    Args:
        text_sample: Text and metadata.
        principles: Principles to apply (None = DEFAULT_PRINCIPLES).

    Returns:
        ConstitutionalResult with the revised text and the change history.
    """
    filt = ConstitutionalFilter(
        principles=principles,
        max_revisions=3,  # up to 3 revisions per principle
    )

    return await filt.apply(
        output=text_sample["text"],
        context=text_sample["context"],
    )


async def main() -> None:
    print("=" * 70)
    print("  Constitutional AI — Dataset: AdvBench + evaluation texts")
    print("=" * 70)

    # ── Part 1: Default principles (P001, P002, P003) ────────────────
    print("\n[Part 1: Default principles]")
    print("  P001 — no_harmful_content (severity: critical)")
    print("  P002 — factual_accuracy   (severity: high)")
    print("  P003 — no_pii_exposure    (severity: high)")
    print()

    for sample in TEST_OUTPUTS[:3]:
        result = await run_constitutional_filter(sample, principles=DEFAULT_PRINCIPLES)
        print_result(sample, result)

    # ── Part 2: Custom principles ────────────────────────────────────
    print("\n[Part 2: Custom principles (P004 + P005)]")
    print("  P004 — no_discriminatory_content (severity: high)")
    print("  P005 — epistemic_humility         (severity: medium)")
    print()

    all_principles = (DEFAULT_PRINCIPLES or []) + CUSTOM_PRINCIPLES
    for sample in TEST_OUTPUTS[3:]:
        result = await run_constitutional_filter(sample, principles=all_principles)
        print_result(sample, result)

    # ── Part 3: Statistical analysis ─────────────────────────────────
    print("\n[Statistical summary]")
    print("  Evaluating all texts with the full set of principles...")

    all_results = []
    for sample in TEST_OUTPUTS:
        r = await run_constitutional_filter(sample, principles=all_principles)
        all_results.append((sample, r))

    total_violations = sum(len(r.revisions) for _, r in all_results)
    fully_safe = sum(1 for _, r in all_results if r.all_principles_satisfied)

    print(f"\n  Texts evaluated         : {len(TEST_OUTPUTS)}")
    print(f"  Fully safe texts        : {fully_safe}/{len(TEST_OUTPUTS)}")
    print(f"  Total violations detected: {total_violations}")
    print(f"  Total revisions applied  : {total_violations}")

    # Group by principle
    from collections import Counter

    violations_by_principle: Counter = Counter()
    for _, r in all_results:
        for rev in r.revisions:
            violations_by_principle[rev.principle_id] += 1

    if violations_by_principle:
        print("\n  Violations per principle:")
        for pid, count in violations_by_principle.most_common():
            print(f"    {pid}: {count} violations")


if __name__ == "__main__":
    asyncio.run(main())


---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()